#### Load data & libraries

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

In [2]:
pd.options.display.max_rows = None
pd.options.display.max_columns = None

In [3]:
# 1. Khởi tạo cấu hình kết nối SparkSession
spark = SparkSession.builder \
    .appName("Feature_Extraction") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
# 2. Đọc dữ liệu trực tiếp từ HDFS
# hdfs_input_path = "hdfs://localhost:9000/big_data/hotel_bookings_cleaned.csv"
# df = spark.read.csv(hdfs_input_path, header=True, inferSchema=True)
# 3. Đọc dữ liệu từ file CSV cục bộ
local_input_path = "hotel_bookings_cleaned.csv"
df_cleaned = spark.read.csv(local_input_path, header=True, inferSchema=True)

In [7]:
df_feature_extracted = df_cleaned \
    .withColumn("total_guests", F.col("adults") + F.col("children") + F.col("babies")) \
    .withColumn("total_nights", F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")) \
    .withColumn(
        "total_estimated_cost", 
        F.when(F.col("total_nights") == 0, F.col("adr"))
         .otherwise(F.col("adr") * F.col("total_nights"))
    ) \
    .withColumn("is_family", F.when((F.col("children") > 0) | (F.col("babies") > 0), 1).otherwise(0)) \
    .withColumn("total_past_bookings", F.col("previous_cancellations") + F.col("previous_bookings_not_canceled")) \
    .withColumn("cancellation_rate", F.col("previous_cancellations") / (F.col("total_past_bookings") + 1))

# Xóa các cột gây nhiễu và Target Leakage
columns_to_drop = ["reservation_status", "reservation_status_date"]
df_feature_extracted = df_feature_extracted.drop(*columns_to_drop)

In [8]:

df_feature_extracted.select("stays_in_weekend_nights","stays_in_week_nights","total_nights", "adr", "total_estimated_cost", "previous_cancellations","previous_bookings_not_canceled", "total_past_bookings","cancellation_rate").show(10)

+-----------------------+--------------------+------------+-----+--------------------+----------------------+------------------------------+-------------------+-----------------+
|stays_in_weekend_nights|stays_in_week_nights|total_nights|  adr|total_estimated_cost|previous_cancellations|previous_bookings_not_canceled|total_past_bookings|cancellation_rate|
+-----------------------+--------------------+------------+-----+--------------------+----------------------+------------------------------+-------------------+-----------------+
|                      0|                   1|           1| 75.0|                75.0|                     0|                             0|                  0|              0.0|
|                      0|                   1|           1| 75.0|                75.0|                     0|                             0|                  0|              0.0|
|                      0|                   2|           2| 98.0|               196.0|                   

#### Data export

In [10]:
df_feature_extracted.write.csv("hotel_bookings_feature_extracted.csv", header=True, mode="overwrite")